Imports

In [1]:
# imports
from esdl import esdl
from esdl.esdl_handler import EnergySystemHandler
import pandas as pd
import numpy as np
import numpy_financial as npf
from decimal import Decimal, ROUND_HALF_UP
import matplotlib.pyplot as plt

In [2]:
# Import the business case runner
from set_up.generic_business_case_runner import *

In [ ]:
# This imports all input variables from EL_input_data.py
from EL_input_data import *

# import all data from ESDL
from NSE_get_data_from_ESDL import *



Active Scenario: most_likely (Index: 1)
All variables saved to NSE_get_data_from_ESDL.pkl


In [4]:
asset_parameters

name,power,efficiency,investment_costs,fixed_opex,variable_opex,wacc
TNVDW,700000000.0,,1750.0,2.25,5.0,8.5
Electrolyzer,500.0,0.6,2000.0,2.0,0.0,8.25
Offtaker,24900000.0,,0.0,0.0,0.0,10.5


Set up calendar_year, business_case_year, operations_years, and decommissioning_years lists

In [5]:
from set_up.construct_timelines import (
    construct_calendar_year_list,
    construct_business_case_year_list,
    construct_operations_years_list,
    construct_decommissioning_years_list
)

Start business case analysis

Construction phase

In [6]:
def construction_phase(**kwargs):
    ''' This function creates a dataframe that contains all cashflows in the construction phase'''
    
    # 1. Get the parameters we need
    capex = kwargs['capex']
    duration_construction = kwargs['duration_construction']

    # 2. Get the timelines
    calendar_years = construct_calendar_year_list(**kwargs)

    # 3. Construct df
    row_names = ['capex', 'total_cashflow_investment']
    df_construction_phase = pd.DataFrame(0.0, index=row_names, columns=calendar_years)
    df_construction_phase.columns.name = "construction_phase"
    
    # 4. Calculate yearly CAPEX value 
    yearly_capex = -capex / duration_construction

    # 5. Add CAPEX numbers to construction years
    df_construction_phase.loc['capex'] = np.where(
        df_construction_phase.columns.isin(construction_years_list),
        yearly_capex, 0.0)

    # 6. Calculate total investments
    # at this moment only one cashflow is in construction phase, can be expanded later
    total_cashflow_investment = df_construction_phase.loc['capex']
    df_construction_phase.loc['total_cashflow_investment'] = total_cashflow_investment
    
    return df_construction_phase

In [7]:
df_construction_phase = construction_phase(**el_parameters)
df_construction_phase.style.format(precision=2)

construction_phase,2027,2028,2029,2030,2031,2032,2033,2034,2035,2036,2037,2038,2039,2040,2041,2042,2043,2044,2045,2046,2047,2048,2049,2050,2051,2052,2053,2054,2055,2056,2057
capex,0.00,-696.33,-696.33,-696.33,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
total_cashflow_investment,0.00,-696.33,-696.33,-696.33,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00


Operational phase

In [8]:
def operational_phase(**kwargs):
    ''' This function creates a dataframe that contains all cashflows in the operational phase'''

    # 1. Get the parameters we need
    inflation = kwargs['inflation']
    opex_df = kwargs['opex_df']
    annual_electricity_costs_ppa = kwargs['annual_electricity_costs_ppa']
    annual_electricity_costs_grid = kwargs['annual_electricity_costs_grid']
    electricity_grid_connection = kwargs['electricity_grid_connection']
    h2_storage_costs = kwargs['h2_storage_costs']
    df_stack_replacement_costs = kwargs['df_stack_replacement_costs']
    hydrogen_revenues = kwargs['hydrogen_revenues']
    hwi_revenues = kwargs['hwi_revenues']

    # 2. Get the timelines
    calendar_years = construct_calendar_year_list(**kwargs)
    operational_years = construct_operations_years_list(**kwargs)
    last_construction_year = construction_years_list[-1]
    business_case_years = construct_business_case_year_list(**kwargs)

    # 3. construct df
    row_names = [
        'opex', 'purchase_electricity_ppa', 'purchase_electricity_grid', 
        'electricity_grid_connection', 'h2_storage_costs', 'stack_replacement', 
        'total_outflow_opex', 'h2_revenues', 'hwi_revenues', 
        'total_revenues_opex', 'net_cashflow_operations'
    ]

    df_operational_phase = pd.DataFrame(0.0, index=row_names, columns=calendar_years)
    df_operational_phase.columns.name = "operational_phase"
    
    # 4. Calculate inflation factors array 
    inf_factors = (1 + inflation) ** business_case_years
    # Turn it into a series to give the calendar years as index
    # This way we can use it in our loop over the operational years
    inf_factor_series = pd.Series(inf_factors, index=calendar_years)

    # 5. Calculate the outflows
    for x in operational_years:

        # Get the inflation factor for that year
        inf_factor = inf_factor_series[x]

        # Outflows
        
        # opex, annual electricity costs ppa/grid are dfs
        df_operational_phase.loc['opex', x] = -opex_df.loc['opex',x] * inf_factor
        df_operational_phase.loc['purchase_electricity_ppa', x] = (
            -annual_electricity_costs_ppa.loc['el_annual_electricity_costs_ppa',x] * 
            inf_factor)
        df_operational_phase.loc['purchase_electricity_grid', x] = (
            -annual_electricity_costs_grid.loc['el_annual_electricity_costs_grid',x] * 
            inf_factor)
        # electricity grid connection is a value    
        df_operational_phase.loc['electricity_grid_connection', x] = (
            -electricity_grid_connection * inf_factor)
        # h2 storage costs is a df
        df_operational_phase.loc['h2_storage_costs', x] = (
            -h2_storage_costs.loc['el_h2_storage_costs',x] * inf_factor)


        # 6. Calculate stack replacement, every 5 years after construction end
        if (x - last_construction_year) % 5 == 0:    
            df_operational_phase.loc['stack_replacement',x] = (
                -df_stack_replacement_costs.loc['stack_replacement', x] * inf_factor)


        # 7. Calculate revenues
        df_operational_phase.loc['h2_revenues', x] = (
            hydrogen_revenues.loc['el_h2_revenues', x] * inf_factor)
        df_operational_phase.loc['hwi_revenues', x] = (
            hwi_revenues.loc['el_hwi_revenues', x] * inf_factor)
    
    
    # 8. Calculate totals
    outflow_rows = [
        'opex', 'purchase_electricity_ppa', 'purchase_electricity_grid', 
        'electricity_grid_connection', 'h2_storage_costs', 'stack_replacement'
    ]
    df_operational_phase.loc['total_outflow_opex'] = df_operational_phase.loc[outflow_rows].sum()

    revenue_rows = ['h2_revenues', 'hwi_revenues']
    df_operational_phase.loc['total_revenues_opex'] = df_operational_phase.loc[revenue_rows].sum()

    # Net cashflow from operation (=EBITDA)
    df_operational_phase.loc['net_cashflow_operations'] = (
        df_operational_phase.loc['total_outflow_opex'] + 
        df_operational_phase.loc['total_revenues_opex'])

    
    return df_operational_phase

In [9]:
df_operational_phase = operational_phase(**el_parameters)
df_operational_phase.style.format(precision=2)

operational_phase,2027,2028,2029,2030,2031,2032,2033,2034,2035,2036,2037,2038,2039,2040,2041,2042,2043,2044,2045,2046,2047,2048,2049,2050,2051,2052,2053,2054,2055,2056,2057
opex,0.00,0.00,0.00,0.00,-164.10,-163.07,-161.94,-160.70,-159.35,-157.87,-156.28,-154.55,-152.70,-150.71,-153.72,-156.79,-159.93,-163.13,-166.39,-169.72,-173.11,-176.58,-180.11,-183.71,-187.38,-191.13,-194.95,-198.85,-202.83,0.00,0.00
purchase_electricity_ppa,0.00,0.00,0.00,0.00,-103.42,-105.44,-107.49,-109.59,-111.73,-113.91,-116.13,-118.39,-120.70,-123.05,-117.26,-111.18,-104.81,-98.14,-91.17,-83.87,-76.25,-68.29,-59.98,-51.31,-52.33,-53.38,-54.45,-55.54,-56.65,0.00,0.00
purchase_electricity_grid,0.00,0.00,0.00,0.00,-10.20,-12.18,-14.24,-16.38,-18.59,-20.89,-23.27,-25.74,-28.30,-30.96,-28.56,-26.06,-23.44,-20.71,-17.86,-14.89,-11.80,-8.57,-5.21,-1.71,-1.75,-1.78,-1.82,-1.86,-1.89,0.00,0.00
electricity_grid_connection,0.00,0.00,0.00,0.00,-31.24,-31.86,-32.50,-33.15,-33.81,-34.49,-35.18,-35.88,-36.60,-37.33,-38.08,-38.84,-39.62,-40.41,-41.22,-42.04,-42.88,-43.74,-44.62,-45.51,-46.42,-47.35,-48.29,-49.26,-50.25,0.00,0.00
h2_storage_costs,0.00,0.00,0.00,0.00,-2.03,-2.07,-2.11,-2.15,-2.20,-2.24,-2.28,-2.33,-2.38,-2.42,-2.53,-2.65,-2.77,-2.89,-3.01,-3.14,-3.27,-3.41,-3.55,-3.69,-3.77,-3.84,-3.92,-4.00,-4.08,0.00,0.00
stack_replacement,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,-457.53,0.00,0.00,0.00,0.00,-486.40,0.00,0.00,0.00,0.00,-537.02,0.00,0.00,0.00,0.00,-592.91,0.00,0.00,0.00,0.00,-654.63,0.00,0.00
total_outflow_opex,0.00,0.00,0.00,0.00,-310.98,-314.63,-318.29,-321.97,-783.21,-329.40,-333.14,-336.90,-340.68,-830.87,-340.15,-335.52,-330.57,-325.28,-856.67,-313.67,-307.32,-300.58,-293.46,-878.85,-291.65,-297.48,-303.43,-309.50,-970.32,0.00,0.00
h2_revenues,0.00,0.00,0.00,0.00,126.46,129.66,132.93,136.28,139.71,143.23,146.83,150.51,154.29,158.15,151.74,145.01,137.94,130.54,122.78,114.66,106.17,97.30,88.02,78.34,79.90,81.50,83.13,84.79,86.49,0.00,0.00
hwi_revenues,0.00,0.00,0.00,0.00,301.01,307.03,313.18,319.44,325.83,332.34,338.99,345.77,352.69,359.74,366.94,374.27,381.76,389.39,397.18,405.13,413.23,421.49,429.92,438.52,447.29,456.24,465.36,474.67,484.16,0.00,0.00
total_revenues_opex,0.00,0.00,0.00,0.00,427.47,436.69,446.10,455.72,465.54,475.57,485.82,496.28,506.97,517.89,518.67,519.28,519.70,519.93,519.96,519.79,519.40,518.79,517.94,516.86,527.19,537.74,548.49,559.46,570.65,0.00,0.00


Decommissioning phase

In [10]:
def decommissioning_phase(**kwargs):
    ''' This function creates a dataframe that contains all cashflows in the decommissioning phase'''
    
    # 1. Get the parameters we need
    capex = kwargs['capex']
    decommissioning_percentage = kwargs['decommissioning_percentage']
    inflation = kwargs['inflation']

    # 2. Get the timelines
    calendar_years = construct_calendar_year_list(**kwargs)
    decommissioning_years = construct_decommissioning_years_list(**kwargs)
    business_case_years = construct_business_case_year_list(**kwargs)

    # 3. construct df
    row_names = ['decommissioning_costs', 'total_cashflow_decommissioning']
    df_decommissioning_phase = pd.DataFrame(0.0, index=row_names, columns=calendar_years)
    df_decommissioning_phase.columns.name = "decommissioning_phase"

    # 4. Calculate inflation factors
    inf_factors = (1 + inflation) ** business_case_years
    # Turn it into a series to give the calendar years as index
    # This way we can use it in our loop over the operational years
    inf_factor_series = pd.Series(inf_factors, index=calendar_years)

    # 5. Calculate decommissioning costs
    total_decommissioning_costs = capex * decommissioning_percentage
    yearly_decommissioning_costs = total_decommissioning_costs / len(decommissioning_years)

    df_decommissioning_phase.loc['decommissioning_costs'] = np.where(
        df_decommissioning_phase.columns.isin(decommissioning_years),
        -yearly_decommissioning_costs * inf_factor_series,
        0.0
    )

    # 4. Calculate totals
    df_decommissioning_phase.loc['total_cashflow_decommissioning'] = df_decommissioning_phase.loc['decommissioning_costs'] 
    
    return df_decommissioning_phase

In [11]:
df_decommissioning_phase = decommissioning_phase(**el_parameters)
df_decommissioning_phase.style.format(precision=2)

decommissioning_phase,2027,2028,2029,2030,2031,2032,2033,2034,2035,2036,2037,2038,2039,2040,2041,2042,2043,2044,2045,2046,2047,2048,2049,2050,2051,2052,2053,2054,2055,2056,2057
decommissioning_costs,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,-37.10,-37.84
total_cashflow_decommissioning,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,-37.10,-37.84


<div class="alert alert-block alert-info">
    <b>Discuss:</b> 
    Which parameters should all be included for the sensitivity analysis? For example, currently decommissioning_percentage is included - but is this really necessary?
    Check also for all other asset business cases
</div>

<div class="alert alert-block alert-info">
    <b>Discuss:</b> 
    Currently the general code blocks below are generated here. Check them across all asset business cases. If they work well for all bc's, I think we can remove them here and only let them be generated by the generic_business_case_runner. That way, the asset working files can be kept nice and short. 
</div>

Taxes & profits - part 1

In [12]:
from finances.taxes_debt_loans import taxes_and_profits_part1

df_taxes_and_profits_part1 = taxes_and_profits_part1(df_operational_phase,**el_parameters)
df_taxes_and_profits_part1.style.format(precision=2)

taxes_and_profits,2027,2028,2029,2030,2031,2032,2033,2034,2035,2036,2037,2038,2039,2040,2041,2042,2043,2044,2045,2046,2047,2048,2049,2050,2051,2052,2053,2054,2055,2056,2057
depreciation,0.00,0.00,0.00,0.00,-83.56,-83.56,-83.56,-83.56,-83.56,-83.56,-83.56,-83.56,-83.56,-83.56,-83.56,-83.56,-83.56,-83.56,-83.56,-83.56,-83.56,-83.56,-83.56,-83.56,-83.56,-83.56,-83.56,-83.56,-83.56,0.00,0.00
ebit,0.00,0.00,0.00,0.00,32.93,38.50,44.25,50.19,-401.23,62.61,69.12,75.82,82.74,-396.53,94.96,100.20,105.58,111.09,-420.27,122.56,128.53,134.64,140.92,-445.55,151.98,156.69,161.50,166.40,-483.23,0.00,0.00


Debt and loan - part 1

In [27]:
from finances.taxes_debt_loans import debt_and_loan_part1

df_debt_and_loan_part1 = debt_and_loan_part1(**el_parameters)
df_debt_and_loan_part1.style.format(precision=2)

debt_and_loan,2027,2028,2029,2030,2031,2032,2033,2034,2035,2036,2037,2038,2039,2040,2041,2042,2043,2044,2045,2046,2047,2048,2049,2050,2051,2052,2053,2054,2055,2056,2057
begin_of_year,0.00,0.00,348.17,696.33,1044.50,996.10,945.27,891.90,835.87,777.03,715.26,650.39,582.28,510.76,435.67,356.83,274.04,187.11,95.84,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
drawdown,0.00,348.17,348.17,348.17,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
repayment_capital,0.00,0.00,0.00,0.00,-48.40,-50.82,-53.37,-56.03,-58.84,-61.78,-64.87,-68.11,-71.52,-75.09,-78.85,-82.79,-86.93,-91.27,-95.84,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,0.00,0.00
end_of_year,0.00,348.17,696.33,1044.50,996.10,945.27,891.90,835.87,777.03,715.26,650.39,582.28,510.76,435.67,356.83,274.04,187.11,95.84,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
repayment_interest,0.00,0.00,0.00,0.00,-52.23,-49.80,-47.26,-44.60,-41.79,-38.85,-35.76,-32.52,-29.11,-25.54,-21.78,-17.84,-13.70,-9.36,-4.79,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,0.00,0.00


Taxes & profits - part 2

In [14]:
from finances.taxes_debt_loans import taxes_and_profits_part2

df_taxes_and_profits_part2 = taxes_and_profits_part2(df_operational_phase,
                                                     df_taxes_and_profits_part1,
                                                     df_debt_and_loan_part1,
                                                     **el_parameters)
df_taxes_and_profits_part2.style.format(precision=2)

taxes_and_profits,2027,2028,2029,2030,2031,2032,2033,2034,2035,2036,2037,2038,2039,2040,2041,2042,2043,2044,2045,2046,2047,2048,2049,2050,2051,2052,2053,2054,2055,2056,2057
depreciation,0.00,0.00,0.00,0.00,-83.56,-83.56,-83.56,-83.56,-83.56,-83.56,-83.56,-83.56,-83.56,-83.56,-83.56,-83.56,-83.56,-83.56,-83.56,-83.56,-83.56,-83.56,-83.56,-83.56,-83.56,-83.56,-83.56,-83.56,-83.56,0.00,0.00
ebit,0.00,0.00,0.00,0.00,32.93,38.50,44.25,50.19,-401.23,62.61,69.12,75.82,82.74,-396.53,94.96,100.20,105.58,111.09,-420.27,122.56,128.53,134.64,140.92,-445.55,151.98,156.69,161.50,166.40,-483.23,0.00,0.00
interest_costs,0.00,0.00,0.00,0.00,-52.23,-49.80,-47.26,-44.60,-41.79,-38.85,-35.76,-32.52,-29.11,-25.54,-21.78,-17.84,-13.70,-9.36,-4.79,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,0.00,0.00
ebt,0.00,0.00,0.00,0.00,-19.29,-11.30,-3.01,5.59,-443.02,23.76,33.35,43.30,53.62,-422.07,73.18,82.36,91.87,101.74,-425.06,122.56,128.53,134.64,140.92,-445.55,151.98,156.69,161.50,166.40,-483.23,0.00,0.00
tax_expenses,0.00,0.00,0.00,0.00,0.00,0.00,0.00,-1.44,0.00,-6.13,-8.60,-11.17,-13.83,0.00,-18.88,-21.25,-23.70,-26.25,0.00,-31.62,-33.16,-34.74,-36.36,0.00,-39.21,-40.43,-41.67,-42.93,0.00,0.00,0.00
net_profits,0.00,0.00,0.00,0.00,-19.29,-11.30,-3.01,4.15,-443.02,17.63,24.75,32.13,39.79,-422.07,54.30,61.11,68.17,75.49,-425.06,90.94,95.37,99.91,104.56,-445.55,112.77,116.27,119.83,123.47,-483.23,0.00,0.00


Debt & loan - part 2

In [15]:
from finances.taxes_debt_loans import debt_and_loan_part2

df_debt_and_loan_part2 = debt_and_loan_part2(df_operational_phase,
                                             df_debt_and_loan_part1,
                                             df_taxes_and_profits_part2,
                                             **el_parameters)

df_debt_and_loan_part2.style.format(precision=2)

debt_and_loan,2027,2028,2029,2030,2031,2032,2033,2034,2035,2036,2037,2038,2039,2040,2041,2042,2043,2044,2045,2046,2047,2048,2049,2050,2051,2052,2053,2054,2055,2056,2057
begin_of_year,0.00,0.00,348.17,696.33,1044.50,996.10,945.27,891.90,835.87,777.03,715.26,650.39,582.28,510.76,435.67,356.83,274.04,187.11,95.84,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
drawdown,0.00,348.17,348.17,348.17,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
repayment_capital,0.00,0.00,0.00,0.00,-48.40,-50.82,-53.37,-56.03,-58.84,-61.78,-64.87,-68.11,-71.52,-75.09,-78.85,-82.79,-86.93,-91.27,-95.84,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,0.00,0.00
end_of_year,0.00,348.17,696.33,1044.50,996.10,945.27,891.90,835.87,777.03,715.26,650.39,582.28,510.76,435.67,356.83,274.04,187.11,95.84,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
repayment_interest,0.00,0.00,0.00,0.00,-52.23,-49.80,-47.26,-44.60,-41.79,-38.85,-35.76,-32.52,-29.11,-25.54,-21.78,-17.84,-13.70,-9.36,-4.79,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,0.00,0.00
cash_flow_for_debt,0.00,0.00,0.00,0.00,116.49,122.06,127.81,132.30,-317.67,140.04,144.07,148.21,152.46,-312.97,159.64,162.51,165.43,168.40,-336.71,174.50,178.93,183.47,188.12,-361.99,196.33,199.83,203.39,207.03,-399.67,0.00,0.00
debt_service,0.00,0.00,0.00,0.00,-100.63,-100.63,-100.63,-100.63,-100.63,-100.63,-100.63,-100.63,-100.63,-100.63,-100.63,-100.63,-100.63,-100.63,-100.63,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,0.00,0.00
cash_after_debt_service,0.00,0.00,0.00,0.00,15.86,21.44,27.19,31.67,-418.30,39.41,43.44,47.58,51.83,-413.60,59.01,61.88,64.80,67.77,-437.34,174.50,178.93,183.47,188.12,-361.99,196.33,199.83,203.39,207.03,-399.67,0.00,0.00


Project reserves

In [16]:
from finances.project_reserves_and_equity import project_reserves

df_project_reserves = project_reserves(**el_parameters)
df_project_reserves.style.format(precision=2)

project_reserves,2027,2028,2029,2030,2031,2032,2033,2034,2035,2036,2037,2038,2039,2040,2041,2042,2043,2044,2045,2046,2047,2048,2049,2050,2051,2052,2053,2054,2055,2056,2057
contingency_injection,-208.90,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
contingency_reserve_balance,208.90,208.90,208.90,208.90,208.90,208.90,208.90,208.90,208.90,208.90,208.90,208.90,208.90,208.90,208.90,208.90,208.90,208.90,208.90,208.90,208.90,208.90,208.90,208.90,208.90,208.90,208.90,208.90,208.90,208.90,0.00
contingency_reserve_to_dividents,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,208.90


<div class="alert alert-block alert-info">
    <b>Discuss:</b> 
    We have a contingency injection that is not used. In real projects, the contingency is always used for something in the project. Does it make sense to payout the unused contingency? I think it may be more logical to 'use' the contingency during the project
</div>

Equity funding

In [17]:
from finances.project_reserves_and_equity import equity_funding

df_equity_funding = equity_funding(df_decommissioning_phase,
               df_debt_and_loan_part2,
               df_construction_phase,
               **el_parameters)

df_equity_funding.style.format(precision=2)

equity_funding,2027,2028,2029,2030,2031,2032,2033,2034,2035,2036,2037,2038,2039,2040,2041,2042,2043,2044,2045,2046,2047,2048,2049,2050,2051,2052,2053,2054,2055,2056,2057
equity_injection,-208.90,-348.17,-348.17,-348.17,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,-37.10,-37.84
dividents_results,0.00,0.00,0.00,0.00,15.86,21.44,27.19,31.67,-418.30,39.41,43.44,47.58,51.83,-413.60,59.01,61.88,64.80,67.77,-437.34,174.50,178.93,183.47,188.12,-361.99,196.33,199.83,203.39,207.03,-399.67,0.00,208.90
equity_cash_flow_result,-208.90,-348.17,-348.17,-348.17,15.86,21.44,27.19,31.67,-418.30,39.41,43.44,47.58,51.83,-413.60,59.01,61.88,64.80,67.77,-437.34,174.50,178.93,183.47,188.12,-361.99,196.33,199.83,203.39,207.03,-399.67,-37.10,171.06


Present value of cash flows

In [18]:
from finances.present_value_and_cumulative_cashflows import present_value_cashflows

df_present_value_cashflows = present_value_cashflows(df_construction_phase,
                                                     df_operational_phase,
                                                     df_decommissioning_phase,
                                                     df_equity_funding,
                                                     **el_parameters)

df_present_value_cashflows.style.format(precision=2)

present_value,2027,2028,2029,2030,2031,2032,2033,2034,2035,2036,2037,2038,2039,2040,2041,2042,2043,2044,2045,2046,2047,2048,2049,2050,2051,2052,2053,2054,2055,2056,2057
sum_net_project_cash_flows,0.00,-696.33,-696.33,-696.33,116.49,122.06,127.81,133.75,-317.67,146.17,152.68,159.38,166.30,-312.97,178.52,183.76,189.14,194.65,-336.71,206.12,212.09,218.20,224.48,-361.99,235.54,240.25,245.06,249.96,-399.67,-37.10,-37.84
present_value_net_cashflows,0.00,-635.92,-580.75,-530.36,81.03,77.54,74.15,70.86,-153.70,64.59,61.61,58.73,55.96,-96.19,50.11,47.10,44.27,41.61,-65.74,36.75,34.53,32.45,30.48,-44.89,26.68,24.85,23.15,21.56,-31.48,-2.67,-2.49
cumulative_value_net_cashflows,0.00,-635.92,-1216.67,-1747.04,-1666.01,-1588.47,-1514.32,-1443.46,-1597.16,-1532.57,-1470.97,-1412.23,-1356.27,-1452.46,-1402.35,-1355.25,-1310.98,-1269.36,-1335.10,-1298.35,-1263.82,-1231.37,-1200.89,-1245.78,-1219.10,-1194.25,-1171.11,-1149.54,-1181.03,-1183.70,-1186.18
present_value_equity_cashflows,-208.90,-317.96,-290.37,-265.18,11.03,13.62,15.77,16.78,-202.38,17.41,17.53,17.53,17.44,-127.12,16.56,15.86,15.17,14.49,-85.38,31.11,29.13,27.28,25.55,-44.89,22.24,20.67,19.21,17.86,-31.48,-2.67,11.24


Cumulative equity and debt cash flows

In [19]:
from finances.present_value_and_cumulative_cashflows import cumulative_equity_and_debt_cashflows

df_cumulative_equity_and_debt_cashflows = cumulative_equity_and_debt_cashflows(df_equity_funding,
                                                                               df_debt_and_loan_part1,
                                                                               **el_parameters)
df_cumulative_equity_and_debt_cashflows.style.format(precision=2)

cumulative equity and debt,2027,2028,2029,2030,2031,2032,2033,2034,2035,2036,2037,2038,2039,2040,2041,2042,2043,2044,2045,2046,2047,2048,2049,2050,2051,2052,2053,2054,2055,2056,2057
cumulative_equity_cashflow,-208.90,-557.07,-905.23,-1253.40,-1237.54,-1216.10,-1188.92,-1157.24,-1575.54,-1536.13,-1492.69,-1445.11,-1393.28,-1806.88,-1747.87,-1685.99,-1621.18,-1553.41,-1990.75,-1816.24,-1637.32,-1453.85,-1265.73,-1627.72,-1431.39,-1231.56,-1028.17,-821.14,-1220.81,-1257.90,-1086.84
cumulative_debt_cashflow,-0.00,-348.17,-696.33,-1044.50,-996.10,-945.27,-891.90,-835.87,-777.03,-715.26,-650.39,-582.28,-510.76,-435.67,-356.83,-274.04,-187.11,-95.84,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00


Discounted cash flows for levelized cost

In [20]:
def discounted_cashflows_for_levelized_cost(df_construction_phase,
                                            df_operational_phase,
                                            df_decommissioning_phase,
                                            df_taxes_and_profits_part2,
                                            df_project_reserves,
                                            **kwargs):
    '''This function creates a dataframe that contains the discounted cashflows for levelized cost calculations'''

    # Using **kwargs we can pass the dictionary of parameters from the input data file into the function here

    # 1. Get the parameters we need
    wacc = kwargs['wacc']

    # 2. Get the timelines
    calendar_years = construct_calendar_year_list(**kwargs)
    business_case_years = construct_business_case_year_list(**kwargs)
    operational_years = construct_operations_years_list(**kwargs)

    # 3. Construct df
    row_names = [
        'capex', 'opex', 'purchase_electricity_ppa', 'purchase_electricity_grid', 
        'electricity_grid_connection', 'h2_storage_costs', 'stack_replacement', 
        'decommissioning', 'interest_costs', 'contingency', 'tax_expenses', 
        'h2_revenues', 'hwi_revenues', 'total_hydrogen_produced'
    ]
    df_discounted_levelized = pd.DataFrame(0.0, index=row_names, columns=calendar_years)
    df_discounted_levelized.columns.name = "discounted cashflows for levelized cost"
    
    # 4. Set up the discount term and calculate levelized costs and revenues
    discount_term = (1 + wacc) ** np.array(business_case_years)

# LEVELIZED COSTS

    df_discounted_levelized.loc['capex'] = df_construction_phase.loc['capex'] / discount_term
    df_discounted_levelized.loc['opex'] = df_operational_phase.loc['opex'] / discount_term
    df_discounted_levelized.loc['purchase_electricity_ppa'] = df_operational_phase.loc['purchase_electricity_ppa'] / discount_term
    df_discounted_levelized.loc['purchase_electricity_grid'] = df_operational_phase.loc['purchase_electricity_grid'] / discount_term
    df_discounted_levelized.loc['electricity_grid_connection'] = df_operational_phase.loc['electricity_grid_connection'] / discount_term
    df_discounted_levelized.loc['h2_storage_costs'] = df_operational_phase.loc['h2_storage_costs'] / discount_term
    df_discounted_levelized.loc['stack_replacement'] = df_operational_phase.loc['stack_replacement'] / discount_term
    df_discounted_levelized.loc['decommissioning'] = df_decommissioning_phase.loc['decommissioning_costs'] / discount_term
    df_discounted_levelized.loc['interest_costs'] = df_taxes_and_profits_part2.loc['interest_costs'] / discount_term

    # contingency (injection + divident return)
    df_discounted_levelized.loc['contingency'] = (
        (df_project_reserves.loc['contingency_injection'] 
         + df_project_reserves.loc['contingency_reserve_to_dividents']) / discount_term)

    # tax expenses
    df_discounted_levelized.loc['tax_expenses'] = df_taxes_and_profits_part2.loc['tax_expenses'] / discount_term


# LEVELIZED REVENUES

    df_discounted_levelized.loc['h2_revenues'] = df_operational_phase.loc['h2_revenues'] / discount_term
    df_discounted_levelized.loc['hwi_revenues'] = df_operational_phase.loc['hwi_revenues'] / discount_term


    # calculate discounted production   
    production = el_h2_sold_to_hpa.loc['el_h2_sold_to_hpa', calendar_years]
    df_discounted_levelized.loc['total_hydrogen_produced'] = np.where(
        production.index.isin(operational_years), 
        production / discount_term, 
        0.0
    )


    return df_discounted_levelized


In [21]:
df_discounted_cashflows_for_levelized_cost = discounted_cashflows_for_levelized_cost(df_construction_phase,
                                                                                     df_operational_phase,
                                                                                     df_decommissioning_phase,
                                                                                     df_taxes_and_profits_part2,
                                                                                     df_project_reserves,
                                                                                     **el_parameters)
df_discounted_cashflows_for_levelized_cost.style.format(precision=2)

discounted cashflows for levelized cost,2027,2028,2029,2030,2031,2032,2033,2034,2035,2036,2037,2038,2039,2040,2041,2042,2043,2044,2045,2046,2047,2048,2049,2050,2051,2052,2053,2054,2055,2056,2057
capex,0.00,-635.92,-580.75,-530.36,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
opex,0.00,0.00,0.00,0.00,-114.14,-103.59,-93.95,-85.14,-77.10,-69.76,-63.06,-56.95,-51.39,-46.32,-43.15,-40.19,-37.44,-34.87,-32.48,-30.26,-28.19,-26.26,-24.46,-22.78,-21.22,-19.77,-18.41,-17.15,-15.98,0.00,0.00
purchase_electricity_ppa,0.00,0.00,0.00,0.00,-71.94,-66.98,-62.36,-58.06,-54.06,-50.33,-46.86,-43.63,-40.62,-37.82,-32.91,-28.50,-24.53,-20.98,-17.80,-14.95,-12.42,-10.15,-8.14,-6.36,-5.93,-5.52,-5.14,-4.79,-4.46,0.00,0.00
purchase_electricity_grid,0.00,0.00,0.00,0.00,-7.09,-7.74,-8.26,-8.68,-9.00,-9.23,-9.39,-9.49,-9.53,-9.51,-8.02,-6.68,-5.49,-4.43,-3.49,-2.66,-1.92,-1.27,-0.71,-0.21,-0.20,-0.18,-0.17,-0.16,-0.15,0.00,0.00
electricity_grid_connection,0.00,0.00,0.00,0.00,-21.73,-20.24,-18.85,-17.56,-16.36,-15.24,-14.20,-13.22,-12.32,-11.47,-10.69,-9.96,-9.27,-8.64,-8.05,-7.50,-6.98,-6.50,-6.06,-5.64,-5.26,-4.90,-4.56,-4.25,-3.96,0.00,0.00
h2_storage_costs,0.00,0.00,0.00,0.00,-1.41,-1.31,-1.22,-1.14,-1.06,-0.99,-0.92,-0.86,-0.80,-0.75,-0.71,-0.68,-0.65,-0.62,-0.59,-0.56,-0.53,-0.51,-0.48,-0.46,-0.43,-0.40,-0.37,-0.34,-0.32,0.00,0.00
stack_replacement,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,-221.37,0.00,0.00,0.00,0.00,-149.49,0.00,0.00,0.00,0.00,-104.84,0.00,0.00,0.00,0.00,-73.53,0.00,0.00,0.00,0.00,-51.57,0.00,0.00
decommissioning,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,-2.67,-2.49
interest_costs,0.00,0.00,0.00,0.00,-36.33,-31.64,-27.42,-23.63,-20.22,-17.17,-14.43,-11.98,-9.80,-7.85,-6.11,-4.57,-3.21,-2.00,-0.94,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,0.00,0.00
contingency,-208.90,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,13.73


Levelized costs

In [ ]:
def levelized_cost_and_revenues(df_discounted_cashflows_for_levelized_cost,
                                **kwargs):
    ''' This function creates a dataframe that contains the levelized costs and revenues'''
    
    # Using **kwargs we can pass the dictionary of parameters from the input data file into the function here

    # 1. Get the row names, except for total_hydrogen_produced
    row_names_levelized_cost = df_discounted_cashflows_for_levelized_cost.index.tolist()
    row_names_levelized_cost.remove('total_hydrogen_produced')

    # 2. Construct df
    df_levelized_cost = pd.DataFrame(0.0, index=row_names_levelized_cost, columns=['cost','revenues'])
    df_levelized_cost.columns.name = "levelized cost calculations"

    # 3. Calculate total discounted hydrogen production
    total_hydrogen_production_sum = df_discounted_cashflows_for_levelized_cost.loc['total_hydrogen_produced'].sum()

    # 4. Calculate the sum for every row
    row_sums = df_discounted_cashflows_for_levelized_cost.loc[row_names_levelized_cost].sum(axis=1)

    # 5. Arrange data over 'revenues' (if positive) and 'costs' (if negative)
    # use factor 1E6 because cost&revenues are in MEUR
    df_levelized_cost['revenues'] = np.where(row_sums>0, row_sums * 1E6 / total_hydrogen_production_sum, 0.0)
    df_levelized_cost['cost'] = np.where(row_sums<0, -row_sums * 1E6 / total_hydrogen_production_sum, 0.0)

    # 6. Calculate total costs and revenues
    total_costs = df_levelized_cost['cost'].sum()
    total_revenues = df_levelized_cost['revenues'].sum()

    # 7. Calculate profits or unprofitable gap
    df_levelized_cost.loc['profits'] = [0.0, 0.0]
    if total_revenues > total_costs:
        df_levelized_cost.loc['profits', 'cost'] = total_revenues - total_costs

    df_levelized_cost.loc['unprofitable_gap'] = [0.0, 0.0]
    if total_costs > total_revenues: 
        df_levelized_cost.loc['unprofitable_gap', 'revenues'] = total_costs - total_revenues


    return df_levelized_cost


In [23]:
df_levelized_cost_and_revenues = levelized_cost_and_revenues(df_discounted_cashflows_for_levelized_cost,
                                                             **el_parameters)
df_levelized_cost_and_revenues.style.format(precision=2)

levelized cost calculations,cost,revenues
capex,133.23,0.00
opex,89.53,0.00
purchase_electricity_ppa,56.07,0.00
purchase_electricity_grid,9.43,0.00
electricity_grid_connection,20.09,0.00
h2_storage_costs,1.38,0.00
stack_replacement,45.82,0.00
decommissioning,0.39,0.00
interest_costs,16.57,0.00
contingency,14.88,0.00


Project KPI's

In [24]:
from finances.kpis import project_kpi

df_project_kpi = project_kpi(df_present_value_cashflows,
                             df_taxes_and_profits_part2,
                             df_construction_phase,
                             **el_parameters)

df_project_kpi.style.format(precision=2)

Project KPIs,Value,Unit
net_present_value,-1186.18,MEUR
internal_rate_of_return,-0.71,%
return_on_investment,-48.44,%
payback_period,-51.61,years
discounted_return_on_investment,-67.60,%
discounted_payback_period,77.16,years


Equity KPI's

In [25]:
from finances.kpis import equity_kpi

df_project_kpi = equity_kpi(df_equity_funding,
                            df_present_value_cashflows,
                             **el_parameters)

df_project_kpi.style.format(precision=2)

Equity KPIs,Value,Unit
net_present_value,-1182.85,MEUR
internal_rate_of_return,-5.11,%
return_of_investment,18.18,%
payback_period,137.51,years
discounted_return_on_investment,-112.48,%
discounted_payback_period,-200.34,years


Output KPI's

In [26]:
from finances.kpis import output_kpi

df_project_kpi = output_kpi(df_levelized_cost_and_revenues,
                            **el_parameters)

df_project_kpi.style.format(precision=2)

Output KPIs,Value,Unit
levelized_cost,393.11,Eur/MWh
levelized_revenues,265.48,Eur/MWh
levelized_profits,-127.63,Eur/MWh
